# Object Detection (YOLOv11) for Complex Engineering Drawings

This notebook is a **step-by-step course** on applying **object detection** to industrial drawings using **Ultralytics YOLOv11** and a **small model (`yolo11n`)** for fast learning. The full workflow uses your local dataset at `data/P-ID Symbols.v1i.yolov11`, then performs inference on both dataset images and `data/Sample-DWG1.pdf`.

## What you will learn

1. What object detection means in engineering drawings (symbols, valves, equipment marks, etc.).
2. How YOLOv11 reads training data in YOLO format (`images` + `labels`).
3. Why dataset folders are split into **train / val / test**, and what each split is used for.
4. How annotations are represented (`class_id x_center y_center width height`).
5. How to train a fast baseline model with `yolo11n`.
6. How to visualize ground truth and prediction results clearly.

## Dataset used in this course

This notebook uses **`data/P-ID Symbols.v1i.yolov11`** already available in your project (or upload to Google Colab).

Important context for learners:
- The annotation boxes in this course are **not generated automatically in this notebook**.
- They were **created manually beforehand** during dataset labeling (human annotation stage), then exported to YOLO format.
- The images in this dataset were also **preprocessed/augmented before export** (as noted by dataset metadata), and this notebook focuses on **training + evaluation + inference** with those prepared files.

## How to run this notebook

Run cells **top to bottom** the first time. Training speed depends on your hardware; the notebook uses a nano model so beginners can run experiments faster.

---
**Prerequisites:** basic Python; optional GPU recommended for training speed.


## Part 1 — Concepts: Object Detection on Engineering Drawings

### 1.1 What is object detection?

Object detection answers: **where** are instances of each class in the image (bounding boxes), and **what** class are they? For drawings, classes might be valves, pumps, instruments, tags, and process symbols.

Outputs are usually **axis-aligned boxes** `(x_center, y_center, width, height)` in normalized image coordinates, plus a **class id** and **confidence**.

### 1.2 Why YOLO?

**YOLO** (You Only Look Once) families use a single CNN forward pass to predict boxes and classes at multiple scales. **YOLOv11** (Ultralytics) continues this line with improved efficiency and accuracy. For teaching and deployment, **`yolo11n` (nano)** is the smallest variant—ideal for **fast iteration**.

### 1.3 What is hard about industrial drawings?

| Challenge | Typical effect | Mitigation ideas |
|-----------|----------------|------------------|
| **Extreme aspect ratios** | Large sheets rasterized to megapixel images | Crop / tile (sliding window), train at moderate `imgsz`, fine-tune on crops |
| **Thin lines** | Low contrast, broken edges after JPEG | Higher DPI rasterization, preprocessing (see your preprocessing notebook), careful augmentation |
| **Dense annotations** | Occlusion, overlapping text | Smaller tiles, specialized classes, higher resolution local crops |
| **Domain shift** | Model trained on one drawing style fails on another | Label a few pages from **your** PDFs and fine-tune |

### 1.4 Annotation format (YOLO)

Each training image has a `.txt` label file with **one line per object**: `class_id x_center y_center width height` (all **normalized** 0–1).


### 1.5 Detection vs. classification vs. segmentation (how they differ)

- **Image classification:** one label for the *entire* image (e.g. "this is a drawing"). It does not tell you *where* symbols are.
- **Object detection:** multiple **boxes** + class labels. This notebook focuses here—ideal for **spotting** balloons, views, title blocks, or symbol regions before OCR or rule-based parsing.
- **Instance / semantic segmentation:** pixel-level masks—more precise boundaries, but **heavier** to label and train. Often used when symbols overlap or shapes are irregular.

### 1.6 How to read evaluation metrics (brief)

During and after training, Ultralytics reports metrics such as **mAP@0.5** (mean Average Precision at IoU threshold 0.5) and **mAP@0.5:0.95** (averaged over stricter IoU thresholds). In simple terms: higher mAP usually means **better agreement** between predicted boxes and ground truth. For engineering QA, you also care about **per-class** behavior (rare symbols may need more data).

### 1.7 Typical industrial workflow (high level)

1. **Rasterize** CAD/PDF to images (control DPI / zoom).
2. **Detect** regions of interest (this notebook).
3. **Crop** each region (optional) and run **OCR**, symbol classifiers, or rule checks.
4. **Iterate:** add failure cases to the training set and **fine-tune**.

### 1.8 Other public resources (for further reading)

- **Floor-plan / CAD symbol spotting:** large CAD datasets (e.g. architectural floor plans) are common benchmarks for **symbol spotting**—layout differs from mechanical drawings but the *pipeline* (tile → detect → post-process) is similar.
- **Research:** recent work combines **detectors** (including YOLO variants) with **VLMs** to parse GD&T, dimensions, and notes from 2D drawings—search for "engineering drawing object detection YOLO" on Google Scholar for up-to-date papers.


### 1.9 AI terms used in Part 1 (for absolute beginners)

If you are new to AI, use this mini-dictionary while reading.

- **AI (Artificial Intelligence):** a broad field where computers perform tasks that normally require human intelligence (recognition, decision support, language understanding).
- **Model:** the mathematical system that learns patterns from data. In this course, YOLOv11 is the model.
- **Training:** the learning process where the model sees labeled examples and adjusts itself.
- **Inference:** using a trained model to make predictions on new images.
- **Dataset:** a collection of images and labels used for training/evaluation.
- **Class:** an object category to detect (for example, a specific valve symbol type).
- **Annotation (label):** human-created ground-truth information that tells the model where objects are and what they are.
- **Bounding box (bbox):** a rectangle around an object.
- **Ground truth:** the correct answer provided by human annotation.
- **Prediction:** the model's output (box + class + confidence).
- **Confidence score:** how sure the model is about a predicted object.
- **YOLO format:** text-label format `class_id x_center y_center width height` with normalized coordinates.
- **Normalized coordinates:** values scaled between 0 and 1, so labels work consistently across different image sizes.
- **Epoch:** one full pass through the training dataset.
- **Batch:** a small group of images processed together in one training step.
- **Loss:** a number showing prediction error during training (lower is generally better).
- **Backpropagation:** the method used to update model weights to reduce loss.
- **Overfitting:** when a model memorizes training data and performs poorly on unseen data.
- **Generalization:** how well the model performs on new, unseen images.
- **Train / Val / Test splits:**
  - **Train:** used to learn (weights are updated).
  - **Val:** used during training to monitor quality (no weight updates).
  - **Test:** used at the end for final unbiased evaluation.
- **mAP (mean Average Precision):** a common object-detection quality metric combining precision-recall behavior across classes.
- **Domain gap:** mismatch between training images and real deployment images (style, resolution, drawing conventions).

Tip for beginners:
When confused, always ask these 3 questions:
1. Is this step for **learning** (train), **checking** (val/test), or **using** (inference)?
2. Is the model seeing **ground truth labels** at this step?
3. Are model weights being **updated** at this step?


## AI Pipeline Overview (Beginner-Friendly)

If you are completely new to AI, think of this project as teaching a student to recognize symbols in engineering drawings.

### 1) Big picture: from raw data to useful prediction

A practical object detection pipeline usually has these stages:

1. **Problem definition**  
   Decide exactly what you want to detect (for example: valve symbols, pumps, instruments) and what success means (for example: high recall so few symbols are missed).

2. **Data collection and labeling**  
   Gather drawing images and create labels. In this course, labels were created **manually beforehand** by human annotators, then saved in YOLO format.

3. **Data preparation**  
   Organize files into `train`, `val`, `test`; verify image-label matching; optionally resize/augment.

4. **Model training**  
   Start from pretrained weights (`yolo11n.pt`) and fine-tune on your labeled dataset.

5. **Validation during training**  
   After each epoch, evaluate on `val` to see whether the model is truly improving on unseen samples.

6. **Final testing**  
   After training decisions are fixed, evaluate on `test` for unbiased final performance.

7. **Inference / deployment**  
   Run the trained model on new drawings (for example `Sample-DWG1.pdf`), visualize outputs, and analyze errors.

8. **Continuous improvement loop**  
   Collect failure cases -> annotate them -> retrain -> retest. This loop is how real AI systems improve over time.

---

### 2) How the model interacts with `train`, `val`, and `test`

This is one of the most important ideas in AI.

- **`train` split**
  - Purpose: learning.
  - What happens: model sees images + labels, computes prediction error, and updates internal weights via backpropagation.
  - In simple words: this is where the model "studies".

- **`val` split**
  - Purpose: monitor generalization while training.
  - What happens: model predicts on `val`, but **does not update weights**.
  - Why important: detects overfitting. If train improves but val stalls/drops, model may be memorizing training data.
  - In simple words: this is like regular quizzes during study.

- **`test` split**
  - Purpose: final exam after all model choices are done.
  - What happens: evaluate once (or very few times) to estimate real-world performance.
  - Rule: do not repeatedly tune your model based on test results, or test becomes biased.
  - In simple words: this is the final exam you should not practice on.

---

### 3) Training-to-inference lifecycle (step-by-step)

1. **Initialize model** from pretrained checkpoint.
2. **Read batch from train** (images and corresponding label text files).
3. **Forward pass**: model predicts boxes/classes.
4. **Compute loss**: compare predictions with ground truth labels.
5. **Backpropagation + optimizer step**: update weights.
6. Repeat for all train batches -> one epoch done.
7. **Run validation** on `val` (no weight update), log metrics (precision/recall/mAP).
8. Keep best checkpoint (`best.pt`) based on validation performance.
9. After training, **evaluate on test** for final unbiased score.
10. Use `best.pt` for **inference on new images/PDF pages** and visualize detections.

---

### 4) Why beginners often get confused (and how to avoid it)

- **Confusion:** "If train accuracy is high, model is good."  
  **Reality:** You must check `val`/`test`; high train alone can be overfitting.

- **Confusion:** "Validation and test are the same."  
  **Reality:** Validation is for development-time decisions; test is for final reporting.

- **Confusion:** "Model created labels automatically."  
  **Reality:** In supervised detection, training labels are human-annotated first; model learns from those examples.

---

### 5) Mental model to remember

- `train` = **learn**
- `val` = **tune/check during learning**
- `test` = **final unbiased check**
- `inference` = **real usage on unseen data**

If you keep this mental model, the rest of the AI workflow becomes much easier to understand.


## Part 2 — Environment setup

Install the core libraries for YOLOv11 training, visualization, and PDF rasterization.


In [1]:
%pip install -q ultralytics opencv-python matplotlib numpy pymupdf pyyaml


Note: you may need to restart the kernel to use updated packages.


  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.
  Consider adding this directory to PATH or, if you prefer to suppress this warning, use --no-warn-script-location.

[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [ ]:
import random
import warnings
from pathlib import Path

import cv2
import matplotlib.pyplot as plt
import numpy as np
import yaml

warnings.filterwarnings("ignore")

try:
    import fitz  # PyMuPDF
except ImportError:
    fitz = None

from ultralytics import YOLO

DATA_YAML = None  # set by local dataset config cell
BEST = None  # set after training

print("Imports OK.")


## Part 3 — Understand your local dataset first (`data/P-ID Symbols.v1i.yolov11`)

Before training, beginners should understand what each folder means.

### Why these folders exist

A YOLO dataset normally has this layout:

- `train/images`, `train/labels` → data used to **learn** model parameters.
- `valid/images`, `valid/labels` → data used during training to **monitor generalization** and tune hyperparameters.
- `test/images`, `test/labels` → data kept separate for **final unbiased evaluation** after model choices are fixed.

### Why split into train / val / test?

If we evaluate only on training images, the model can appear excellent by memorizing data. Splits prevent this:

- **Train** answers: "Can the model fit known examples?"
- **Val** answers: "Does performance improve on unseen data while training?"
- **Test** answers: "How good is the final model in a fair, real-world-like check?"

### About annotations in this course

The bounding boxes used here were **manually annotated by humans in a previous labeling stage** and exported in YOLO format. In this notebook, we **do not create annotations**; we use these already-prepared labels to train and evaluate YOLOv11.


In [ ]:
# Local dataset root (already in this project)
NOTEBOOK_DIR = Path(".").resolve()
DATASET_ROOT = NOTEBOOK_DIR / "data" / "P-ID Symbols.v1i.yolov11"
DATA_YAML = DATASET_ROOT / "data.yaml"

if not DATASET_ROOT.exists():
    raise FileNotFoundError(f"Dataset folder not found: {DATASET_ROOT}")
if not DATA_YAML.exists():
    raise FileNotFoundError(f"data.yaml not found: {DATA_YAML}")

print("Dataset root:", DATASET_ROOT)
print("data.yaml:", DATA_YAML)


### Read `data.yaml` and explain what it controls (line by line)

`data.yaml` is the dataset configuration file YOLO uses to understand your data.

In this project, the important keys are:

- `train`
  - Path to training images.
  - YOLO uses these samples to learn and update model weights.

- `val`
  - Path to validation images.
  - YOLO checks performance here during training (without updating weights).

- `test`
  - Path to test images.
  - Used for final, less-biased evaluation after training decisions are fixed.

- `nc`
  - Number of classes in dataset.

- `names`
  - Ordered class-name list.
  - Index in this list is the `class_id` used in label files.

How `data.yaml` connects to labels:
- For each image in `images/`, YOLO expects a `.txt` file with the same stem in `labels/`.
- Each row in that `.txt` uses `class_id`, and YOLO maps it to class text using `names[class_id]`.

Must-check rules for beginners:
1. `len(names) == nc`
2. Class IDs in label files must be within `0..nc-1`
3. Train/val/test paths must exist and contain images


In [ ]:
cfg = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))

print("=== Original data.yaml ===")
print(DATA_YAML.read_text(encoding="utf-8"))
print()

print("Number of classes (nc):", cfg.get("nc"))
names = cfg.get("names", [])
print("First 15 class names:", names[:15])

split_dirs = {
    "train": (DATASET_ROOT / "train" / "images").resolve(),
    "val": (DATASET_ROOT / "valid" / "images").resolve(),
    "test": (DATASET_ROOT / "test" / "images").resolve(),
}

for split_name, img_dir in split_dirs.items():
    lbl_dir = Path(str(img_dir).replace("images", "labels"))
    image_files = [p for p in img_dir.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}]
    label_files = list(lbl_dir.glob("*.txt")) if lbl_dir.exists() else []

    print(f"[{split_name}] images={len(image_files)} labels={len(label_files)}")
    print(f"  image dir: {img_dir}")
    print(f"  label dir: {lbl_dir}")

# Build a resolved YAML with absolute split paths for robust training.
resolved_cfg = {
    "path": str(DATASET_ROOT.resolve()),
    "train": str(split_dirs["train"]),
    "val": str(split_dirs["val"]),
    "test": str(split_dirs["test"]),
    "nc": cfg["nc"],
    "names": cfg["names"],
}
DATA_YAML_RESOLVED = DATASET_ROOT / "data.resolved.yaml"
DATA_YAML_RESOLVED.write_text(yaml.safe_dump(resolved_cfg, sort_keys=False, allow_unicode=True), encoding="utf-8")
DATA_YAML = DATA_YAML_RESOLVED
print("\nUsing resolved training config:", DATA_YAML)


### Dataset provenance and preprocessing notes

This dataset includes metadata files (`README.dataset.txt`, `README.metadata.txt`) that describe how the dataset was exported.

From those notes, you can explain to learners that:
- source annotations were prepared beforehand (human-labeled stage),
- images were resized to `640x640`,
- additional augmented copies were generated (flip, rotate, crop, brightness, noise).

So this notebook uses **already prepared training data** (images + manual annotation labels) to focus on YOLOv11 training and inference.

In [ ]:
meta_file = DATASET_ROOT / "README.dataset.txt"

if meta_file.exists():
    lines = meta_file.read_text(encoding="utf-8", errors="ignore").splitlines()
    print("=== Dataset metadata preview ===")
    print("\n".join(lines[:35]))
else:
    print(f"Metadata file not found: {meta_file}")


### Annotation files in `labels/`: what each file, line, and column means

Each image in `images/` should have a matching label file in `labels/` with the same filename stem.

Example mapping:
- image: `train/images/example_001.jpg`
- label: `train/labels/example_001.txt`

Inside `example_001.txt`, each **line = one object instance**:

`class_id x_center y_center width height`

Column-by-column meaning:

1. `class_id`
   - Integer class index (e.g., `0`, `1`, `2`, ...).
   - Maps directly to `names[class_id]` in `data.yaml`.

2. `x_center`
   - Horizontal center of bounding box, normalized by image width.
   - Range is typically `0` to `1`.

3. `y_center`
   - Vertical center of bounding box, normalized by image height.
   - Range is typically `0` to `1`.

4. `width`
   - Bounding-box width normalized by image width.

5. `height`
   - Bounding-box height normalized by image height.

Row-level meaning:
- 1 line = 1 annotated symbol/object in that image.
- If one image has 10 objects, its label file has 10 lines.
- If an image has no objects of target classes, the label file may be empty or absent depending on export settings.

These labels are **ground-truth labels created by human annotation in advance**. They are the "correct answers" used during training.


In [ ]:
def preview_label_lines(split="train", max_files=3):
    lbl_dir = DATASET_ROOT / split / "labels"
    files = sorted(lbl_dir.glob("*.txt"))[:max_files]
    if not files:
        print(f"No label files found in {lbl_dir}")
        return

    print(f"Showing first label lines from '{split}' split:")
    for fp in files:
        lines = fp.read_text(encoding="utf-8").splitlines()
        print("-", fp.name)
        for ln in lines[:3]:  # first 3 objects in that image
            print("   ", ln)

preview_label_lines("train", max_files=3)


In [ ]:
# Parse one real label row to make columns concrete for beginners.
def explain_one_label_row(split="train"):
    cfg_local = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
    names_local = cfg_local["names"]

    lbl_dir = DATASET_ROOT / split / "labels"
    txt_files = sorted(lbl_dir.glob("*.txt"))
    if not txt_files:
        print(f"No label files in {lbl_dir}")
        return

    # Find first non-empty label file
    target = None
    for fp in txt_files:
        lines = [ln.strip() for ln in fp.read_text(encoding="utf-8").splitlines() if ln.strip()]
        if lines:
            target = (fp, lines[0])
            break

    if target is None:
        print(f"All label files are empty in {lbl_dir}")
        return

    fp, row = target
    parts = row.split()
    class_id = int(parts[0])
    x_center, y_center, width, height = map(float, parts[1:5])

    print("Label file:", fp.name)
    print("Raw row:", row)
    print("class_id:", class_id, "->", names_local[class_id])
    print("x_center:", x_center)
    print("y_center:", y_center)
    print("width:", width)
    print("height:", height)

explain_one_label_row("train")


### Ground-truth labels vs Predicted labels (and where they come from)

This distinction is one of the most important ideas in AI.

#### Ground-truth labels

- **What they are**: the correct answers for each image.
- **Where they come from**: manual annotation by humans before training.
- **Where they are stored**: `.txt` files in `train/labels`, `valid/labels`, `test/labels`.
- **What they contain**: rows in YOLO format (`class_id x_center y_center width height`).

#### Predicted labels

- **What they are**: model outputs (its guesses).
- **Where they come from**: forward pass of trained YOLO model.
- **Where they are stored**: usually in memory as prediction objects; optionally exported as files/images.
- **What they contain**: predicted box coordinates, predicted class, and confidence score.

#### How they interact in training

1. Model reads image and creates predicted labels.
2. Training code compares prediction with ground truth labels.
3. The mismatch becomes loss.
4. Optimizer updates weights to reduce that loss.
5. Repeat for many batches/epochs.

#### During inference on new images

- Typically you only have predicted labels.
- Ground truth usually does not exist yet.
- If you want quantitative evaluation on new images, you must manually annotate them first.

Simple mental model:
- Ground truth = teacher's answer key.
- Prediction = student's answer.
- Training = checking answers and improving after each mistake.


### Export dataset images with drawn annotations (saved to another folder)

This step reads image-label pairs, draws YOLO bounding boxes on each image, and saves the annotated results to a separate output directory.

- Output root: `outputs/part3_labeled_images/`
- Output structure: one subfolder per split (`train`, `valid`, `test`)
- You can control how many images to export per split.
  - Set `MAX_IMAGES_PER_SPLIT = None` to export **all** images.


In [ ]:
# Draw ground-truth annotations and save annotated images to another folder.
OUTPUT_ROOT = NOTEBOOK_DIR / "outputs" / "part3_labeled_images"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

# Export all images in each split.
MAX_IMAGES_PER_SPLIT = None

cfg_local = yaml.safe_load(DATA_YAML.read_text(encoding="utf-8"))
class_names_local = cfg_local["names"]


def draw_yolo_labels_on_bgr(bgr, label_file: Path, class_names):
    h, w = bgr.shape[:2]
    vis = bgr.copy()
    if not label_file.exists():
        return vis

    for raw in label_file.read_text(encoding="utf-8").splitlines():
        parts = raw.strip().split()
        if len(parts) < 5:
            continue

        cls = int(parts[0])
        xc, yc, bw, bh = map(float, parts[1:5])
        x1 = int((xc - bw / 2) * w)
        y1 = int((yc - bh / 2) * h)
        x2 = int((xc + bw / 2) * w)
        y2 = int((yc + bh / 2) * h)

        color = (0, 255, 0)
        cv2.rectangle(vis, (x1, y1), (x2, y2), color, 2)
        cls_name = class_names[cls] if 0 <= cls < len(class_names) else str(cls)
        cv2.putText(vis, cls_name, (x1, max(y1 - 6, 0)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)

    return vis


summary = {}
for split in ["train", "valid", "test"]:
    img_dir = DATASET_ROOT / split / "images"
    lbl_dir = DATASET_ROOT / split / "labels"
    out_dir = OUTPUT_ROOT / split
    out_dir.mkdir(parents=True, exist_ok=True)

    images = sorted([p for p in img_dir.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}])
    if MAX_IMAGES_PER_SPLIT is not None:
        images = images[:MAX_IMAGES_PER_SPLIT]

    saved = 0
    for img_path in images:
        bgr = cv2.imread(str(img_path))
        if bgr is None:
            continue

        label_path = lbl_dir / f"{img_path.stem}.txt"
        annotated = draw_yolo_labels_on_bgr(bgr, label_path, class_names_local)

        out_path = out_dir / img_path.name
        cv2.imwrite(str(out_path), annotated)
        saved += 1

    summary[split] = saved

print("Saved annotated images to:", OUTPUT_ROOT)
for split, count in summary.items():
    print(f"  {split}: {count} image(s)")

# Show a quick preview from exported images
for split in ["train", "valid", "test"]:
    out_dir = OUTPUT_ROOT / split
    sample = sorted(out_dir.glob("*"))[:1]
    if not sample:
        continue

    img = cv2.imread(str(sample[0]))
    img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(10, 7))
    plt.imshow(img)
    plt.title(f"Saved annotated preview - {split}: {sample[0].name}")
    plt.axis("off")
    plt.show()


## Part 4 — Visualize ground-truth labels

We draw boxes from YOLO label files so you **see** what the model is asked to learn.


In [ ]:
def yolo_line_to_box(line: str, w: int, h: int):
    """Convert one YOLO label line to pixel xyxy."""
    parts = line.strip().split()
    if len(parts) < 5:
        return None
    cls = int(parts[0])
    xc, yc, bw, bh = map(float, parts[1:5])
    x1 = int((xc - bw / 2) * w)
    y1 = int((yc - bh / 2) * h)
    x2 = int((xc + bw / 2) * w)
    y2 = int((yc + bh / 2) * h)
    return cls, (x1, y1, x2, y2)


def load_class_names(yaml_path: Path):
    cfg = yaml.safe_load(yaml_path.read_text(encoding="utf-8"))
    names = cfg.get("names")
    if isinstance(names, dict):
        names = [names[k] for k in sorted(names.keys(), key=lambda x: int(x))]
    return names


def show_labeled_image(image_path: Path, label_path: Path, class_names, title=None):
    bgr = cv2.imread(str(image_path))
    if bgr is None:
        print("Failed to read", image_path)
        return
    rgb = cv2.cvtColor(bgr, cv2.COLOR_BGR2RGB)
    h, w = rgb.shape[:2]
    if label_path.exists():
        for line in label_path.read_text(encoding="utf-8").splitlines():
            parsed = yolo_line_to_box(line, w, h)
            if not parsed:
                continue
            cls, (x1, y1, x2, y2) = parsed
            color = (255, 0, 0) if cls == 0 else (0, 180, 255)
            cv2.rectangle(rgb, (x1, y1), (x2, y2), color, 2)
            name = class_names[cls] if cls < len(class_names) else str(cls)
            cv2.putText(rgb, name, (x1, max(y1 - 4, 0)), cv2.FONT_HERSHEY_SIMPLEX, 0.5, color, 1, cv2.LINE_AA)
    plt.figure(figsize=(12, 8))
    plt.imshow(rgb)
    plt.axis("off")
    plt.title(title or image_path.name)
    plt.show()


In [ ]:
if DATA_YAML is not None and DATA_YAML.exists():
    class_names = load_class_names(DATA_YAML)

    # Visualize each split so beginners can see all subsets directly.
    split_dirs = {
        "train": DATASET_ROOT / "train" / "images",
        "valid": DATASET_ROOT / "valid" / "images",
        "test": DATASET_ROOT / "test" / "images",
    }

    random.seed(42)
    for split_name, img_dir in split_dirs.items():
        lbl_dir = DATASET_ROOT / split_name / "labels"
        imgs = sorted([p for p in img_dir.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}])

        print(f"\n=== {split_name.upper()} split ===")
        print(f"Images folder: {img_dir}")
        print(f"Labels folder: {lbl_dir}")
        print(f"Total images: {len(imgs)}")

        sample = random.sample(imgs, min(2, len(imgs)))
        for p in sample:
            lbl = lbl_dir / (p.stem + ".txt")
            show_labeled_image(p, lbl, class_names, title=f"GT ({split_name}): {p.name}")
else:
    print("Skip visualization until dataset is available.")


### Before you train (practical notes)

- **Nano model (`yolo11n.pt`):** fewest parameters → fastest training and inference; use larger models when accuracy plateaus.
- **Epochs:** `40` is a teaching default. If mAP is still climbing at the end, **increase epochs** or adjust learning rate (see Ultralytics docs).
- **Batch size:** if you see CUDA out-of-memory, **lower `BATCH`** (e.g. 4 or 2).
- **Augmentations:** Ultralytics enables sensible defaults (mosaic, flip, scale, etc.). For drawings with **critical tiny text**, overly strong blur or color jitter may hurt—after you understand defaults, you can tune `augment` flags in `model.train(...)`.
- **Reproducibility:** set `seed=` in `train()` if you need repeatable runs.


## Part 5 — Train YOLOv11n (small model, short run)

We use **`yolo11n.pt`** (nano). Training hyperparameters are chosen for **speed**; increase `epochs` and use larger models (`yolo11s`, `m`, ...) when you need better accuracy.

| Parameter | Typical meaning |
|-----------|-----------------|
| `epochs` | How many full passes over the training set |
| `imgsz` | Letterbox resize size (e.g. 640) |
| `batch` | Mini-batch size (reduce if you hit OOM) |
| `device` | `0` for first GPU, `cpu` otherwise |


In [ ]:
# Auto-pick device
import torch
DEVICE = "0" if torch.cuda.is_available() else "cpu"
print("Using device:", DEVICE)


In [ ]:
EPOCHS = 40  # increase (e.g. 100+) for better accuracy when you have time
IMGSZ = 640
BATCH = 8 if DEVICE != "cpu" else 4
PROJECT = Path("runs/detect")
RUN_NAME = "eng_draw_yolo11n"

if DATA_YAML is not None and DATA_YAML.exists():
    model = YOLO("yolo11n.pt")
    results = model.train(
        data=str(DATA_YAML),
        epochs=EPOCHS,
        imgsz=IMGSZ,
        batch=BATCH,
        device=DEVICE,
        project=str(PROJECT),
        name=RUN_NAME,
        exist_ok=True,
        save_period=1,  # save epoch checkpoints for Part 9 comparison
        verbose=True,
    )
    BEST = PROJECT / RUN_NAME / "weights" / "best.pt"
    print("Best weights:", BEST)
else:
    BEST = None
    print("Training skipped — dataset not available.")


## Part 6 - Full training metrics and interpretation

In this section, we review metrics from multiple views:

1. **Direct validation output** from `model.val(...)`.
2. **Training curves** in `results.png` (loss/precision/recall/mAP across epochs).
3. **Class confusion** in `confusion_matrix.png`.

Where files are usually saved:
- Common default run: `runs/detect/train/`
- In this notebook run: `runs/detect/eng_draw_yolo11n/`

The code below auto-detects the current run directory and displays all available metrics artifacts.


In [ ]:
if BEST is not None and Path(BEST).exists():
    model = YOLO(str(BEST))
    metrics = model.val(data=str(DATA_YAML), imgsz=IMGSZ, device=DEVICE)
    print(metrics)

    run_dir = PROJECT / RUN_NAME
    # Fallback for default Ultralytics run naming (e.g., runs/detect/train)
    if not run_dir.exists():
        fallback_dir = Path("runs/detect/train")
        run_dir = fallback_dir if fallback_dir.exists() else run_dir

    print("\nRun directory:", run_dir)

    results_png = run_dir / "results.png"
    confusion_png = run_dir / "confusion_matrix.png"
    csv_path = run_dir / "results.csv"

    # Display training curves if available
    if results_png.exists():
        img = cv2.imread(str(results_png))
        img = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(16, 10))
        plt.imshow(img)
        plt.title("Training curves (loss, precision, recall, mAP)")
        plt.axis("off")
        plt.show()
    else:
        print("results.png not found:", results_png)

    # Display confusion matrix if available
    if confusion_png.exists():
        cm = cv2.imread(str(confusion_png))
        cm = cv2.cvtColor(cm, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(10, 10))
        plt.imshow(cm)
        plt.title("Confusion matrix")
        plt.axis("off")
        plt.show()
    else:
        print("confusion_matrix.png not found:", confusion_png)

    # Print last-epoch metric row from results.csv (if present)
    if csv_path.exists():
        import csv
        with open(csv_path, "r", encoding="utf-8") as f:
            rows = list(csv.DictReader(f))

        if rows:
            last = rows[-1]
            print("\nLast epoch summary from results.csv (selected key metrics):")
            keys_of_interest = [
                "epoch",
                "train/box_loss", "train/cls_loss", "train/dfl_loss",
                "val/box_loss", "val/cls_loss", "val/dfl_loss",
                "metrics/precision(B)", "metrics/recall(B)",
                "metrics/mAP50(B)", "metrics/mAP50-95(B)",
            ]
            for k in keys_of_interest:
                if k in last:
                    print(f"  {k}: {last[k]}")

            print("\nAll metrics available in last epoch row:")
            for k, v in last.items():
                print(f"  {k}: {v}")
        else:
            print("results.csv exists but has no rows.")
    else:
        print("results.csv not found:", csv_path)
else:
    print("No trained weights to validate.")


### How to interpret these metrics (detailed beginner guide)

#### 1) Loss metrics (`train/box_loss`, `train/cls_loss`, `train/dfl_loss`, and validation counterparts)

- **What they mean**
  - `box_loss`: how far predicted boxes are from ground-truth boxes.
  - `cls_loss`: how often predicted classes are wrong.
  - `dfl_loss`: distribution focal loss used by YOLO box regression.

- **Good vs bad (practical view)**
  - Lower is generally better.
  - A good sign is train and val losses both decreasing and then stabilizing.
  - A warning sign is train loss keeps decreasing but val loss increases (possible overfitting).

#### 2) Precision and Recall

- **Precision** = among predicted objects, how many are correct.
  - High precision means fewer false positives.
- **Recall** = among true objects, how many are detected.
  - High recall means fewer missed objects.

- **Good vs bad**
  - Good detector needs balance. Extremely high precision with low recall often means model is too conservative.
  - In engineering QA, low recall can be risky because missing symbols may break downstream logic.

#### 3) mAP metrics (`mAP50`, `mAP50-95`)

- `mAP50`: quality at IoU threshold 0.50 (more forgiving).
- `mAP50-95`: average quality over stricter IoU thresholds (harder metric).

- **Good vs bad**
  - Higher is better for both.
  - `mAP50-95` is usually lower than `mAP50`; this is normal.
  - Track trend over epochs rather than a single number.

#### 4) Confusion matrix

- Rows/columns represent true vs predicted classes.
- Strong diagonal = many correct class predictions.
- Off-diagonal hotspots = classes the model confuses.

Use confusion matrix to decide:
- which classes need more data,
- which labels may be inconsistent,
- where class definitions are too similar.

---

### Overfitting and underfitting (must-know)

- **Overfitting**
  - Model memorizes training data but generalizes poorly.
  - Typical signal: train loss keeps improving while val metrics stop improving or worsen.
  - Fix ideas: add/clean data, stronger augmentation, regularization, early stopping, reduce model complexity.

- **Underfitting**
  - Model has not learned enough patterns.
  - Typical signal: both train and val metrics stay low; losses remain high.
  - Fix ideas: train longer, improve labels, increase model capacity, tune hyperparameters.

If learners want deeper study, suggest topics:
- precision-recall curve analysis,
- class imbalance handling,
- confidence threshold tuning,
- calibration and error analysis pipeline.


## Part 7 — Inference on validation images (visual)

We run prediction on a few **validation** images and display **`results.plot()`** outputs (BGR→RGB).


In [ ]:
def show_predictions(model: YOLO, image_paths, conf=0.25, max_images=6):
    image_paths = list(image_paths)[:max_images]
    for p in image_paths:
        results = model.predict(source=str(p), conf=conf, verbose=False)
        im = results[0].plot()  # BGR
        im_rgb = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
        plt.figure(figsize=(12, 8))
        plt.imshow(im_rgb)
        plt.axis("off")
        plt.title(f"Pred: {Path(p).name}")
        plt.show()


if BEST is not None and Path(BEST).exists():
    val_img_dir = DATASET_ROOT / "valid" / "images"
    val_imgs = sorted([p for p in val_img_dir.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}])
    model = YOLO(str(BEST))
    random.seed(42)
    sample = random.sample(val_imgs, min(4, len(val_imgs)))
    show_predictions(model, sample, conf=0.25)
else:
    print("Inference skipped — train the model first.")


## Part 8 — Test-set baseline vs rasterized-page inference

In this part we run two steps so learners can compare model behavior clearly:

1. **Step A (in-domain):** inference on `test/images`.
   - These images are unseen during training, but still from the same dataset style.
   - Results are usually more stable and accurate.

2. **Step B (out-of-domain):** inference on rasterized `data/Sample-DWG1.pdf` page.
   - This page may differ in style, symbol density, rendering quality, and object scale.
   - Results are often less accurate than Step A.

Learning goal:
- Understand why a model can perform well on test split but degrade on real external documents.


In [ ]:
# Inference on test split images (visual baseline)
if BEST is not None and Path(BEST).exists():
    test_img_dir = DATASET_ROOT / "test" / "images"
    test_imgs = sorted([p for p in test_img_dir.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}])

    model = YOLO(str(BEST))
    random.seed(7)
    sample_test = random.sample(test_imgs, min(4, len(test_imgs)))

    print(f"Test images available: {len(test_imgs)}")
    print("Showing predictions on test images (in-domain baseline)...")
    show_predictions(model, sample_test, conf=0.25)
else:
    print("Inference on test split skipped — train the model first.")


In [ ]:
NOTEBOOK_DIR = Path(".").resolve()
PDF_PATH = NOTEBOOK_DIR / "data" / "Sample-DWG1.pdf"
OUT_DIR = NOTEBOOK_DIR / "runs" / "inference_pdf"
OUT_DIR.mkdir(parents=True, exist_ok=True)


def pdf_page_to_png(pdf_path: Path, page_index: int = 0, zoom: float = 2.0) -> Path:
    if fitz is None:
        raise RuntimeError("PyMuPDF (pymupdf) not available")
    doc = fitz.open(pdf_path)
    page = doc.load_page(page_index)
    mat = fitz.Matrix(zoom, zoom)
    pix = page.get_pixmap(matrix=mat, alpha=False)
    out_path = OUT_DIR / f"{pdf_path.stem}_page{page_index}_z{zoom}.png"
    pix.save(out_path.as_posix())
    doc.close()
    return out_path


In [ ]:
if not PDF_PATH.exists():
    print("PDF not found at:", PDF_PATH)
elif BEST is None or not Path(BEST).exists():
    print("Train the model first to run PDF inference.")
elif fitz is None:
    print("PyMuPDF missing.")
else:
    png_path = pdf_page_to_png(PDF_PATH, page_index=0, zoom=2.0)
    print("Rasterized:", png_path)
    model = YOLO(str(BEST))
    results = model.predict(source=str(png_path), conf=0.15, imgsz=1280, verbose=False)
    im = results[0].plot()
    im_rgb = cv2.cvtColor(im, cv2.COLOR_BGR2RGB)
    plt.figure(figsize=(16, 12))
    plt.imshow(im_rgb)
    plt.axis("off")
    plt.title("Predictions on Sample-DWG1.pdf (page 0, rasterized)")
    plt.show()
    # Side-by-side: original raster vs predictions
    base = cv2.imread(str(png_path))
    base_rgb = cv2.cvtColor(base, cv2.COLOR_BGR2RGB)
    fig, axes = plt.subplots(1, 2, figsize=(20, 10))
    axes[0].imshow(base_rgb)
    axes[0].set_title("Input (rasterized PDF)")
    axes[0].axis("off")
    axes[1].imshow(im_rgb)
    axes[1].set_title("YOLO predictions")
    axes[1].axis("off")
    plt.tight_layout()
    plt.show()


### Interpreting the comparison: test images vs rasterized PDF page

After running Step A and Step B, learners usually observe that predictions on test images look better than on the rasterized page.

Common reasons for lower accuracy on rasterized page:

- **Domain shift:** training/test images and real PDF raster look different.
- **Scale mismatch:** symbols in full-sheet raster can be much smaller than symbols seen during training.
- **Rendering artifacts:** thin lines may blur or break during PDF rasterization.
- **Layout complexity:** real drawings can be denser, with overlaps and clutter not equally represented in training data.
- **Class coverage gap:** some symbol variants in real page may be rare or absent in training labels.

What to do for better results:

1. **Label a small real subset** from your own PDFs and fine-tune the model.
2. **Use tiling/sliding-window inference** on large pages so symbols appear larger to the detector.
3. **Tune rasterization quality** (higher zoom/DPI) before inference.
4. **Increase training quality/time** (more epochs, better augmentation design, error-focused relabeling).
5. **Run error analysis loop**: inspect false positives/false negatives -> add targeted labels -> retrain.

Key takeaway for beginners:
- Good results on dataset test split do not automatically guarantee good performance on production documents.
- Bridging this gap is a normal and essential part of real AI projects.


## Part 9 — Compare early training checkpoint vs best checkpoint

In this part, we compare two checkpoints from the **same training run**:

- **Early checkpoint** (one of the first epochs): model has just started learning.
- **Best checkpoint (`best.pt`)**: model state with best validation performance.

Why this is useful for beginners:
- You can visually see how predictions improve during training.
- You can understand that training is a gradual process, not an instant jump to good results.
- You can inspect remaining mistakes even in `best.pt`.


In [ ]:
weights_dir = PROJECT / RUN_NAME / "weights"

if not weights_dir.exists() or BEST is None or not Path(BEST).exists():
    print("Please train the model first so checkpoints are available.")
else:
    # Find earliest saved epoch checkpoint. With save_period=1, files look like epoch1.pt, epoch2.pt, ...
    epoch_ckpts = sorted(weights_dir.glob("epoch*.pt"), key=lambda p: int(p.stem.replace("epoch", "")) if p.stem.replace("epoch", "").isdigit() else 10**9)

    if epoch_ckpts:
        early_ckpt = epoch_ckpts[0]
    else:
        # Fallback if epoch checkpoints are not present (older run config)
        early_ckpt = weights_dir / "last.pt"

    if not early_ckpt.exists():
        raise FileNotFoundError("No early checkpoint found. Re-run training with save_period=1.")

    print("Early checkpoint:", early_ckpt)
    print("Best checkpoint:", BEST)

    early_model = YOLO(str(early_ckpt))
    best_model = YOLO(str(BEST))

    # --- A) Compare on test images ---
    test_img_dir = DATASET_ROOT / "test" / "images"
    test_imgs = sorted([p for p in test_img_dir.glob("*") if p.suffix.lower() in {".jpg", ".jpeg", ".png", ".bmp"}])

    if test_imgs:
        random.seed(11)
        compare_imgs = random.sample(test_imgs, min(3, len(test_imgs)))

        for p in compare_imgs:
            early_res = early_model.predict(source=str(p), conf=0.25, verbose=False)[0]
            best_res = best_model.predict(source=str(p), conf=0.25, verbose=False)[0]

            early_rgb = cv2.cvtColor(early_res.plot(), cv2.COLOR_BGR2RGB)
            best_rgb = cv2.cvtColor(best_res.plot(), cv2.COLOR_BGR2RGB)

            fig, axes = plt.subplots(1, 2, figsize=(18, 8))
            axes[0].imshow(early_rgb)
            axes[0].set_title(f"Early checkpoint: {p.name}")
            axes[0].axis("off")

            axes[1].imshow(best_rgb)
            axes[1].set_title(f"Best checkpoint: {p.name}")
            axes[1].axis("off")
            plt.tight_layout()
            plt.show()

    # --- B) Compare on rasterized PDF page ---
    if PDF_PATH.exists() and fitz is not None:
        png_path = OUT_DIR / f"{PDF_PATH.stem}_page0_z2.0.png"
        if not png_path.exists():
            png_path = pdf_page_to_png(PDF_PATH, page_index=0, zoom=2.0)

        early_pdf = early_model.predict(source=str(png_path), conf=0.15, imgsz=1280, verbose=False)[0]
        best_pdf = best_model.predict(source=str(png_path), conf=0.15, imgsz=1280, verbose=False)[0]

        early_pdf_rgb = cv2.cvtColor(early_pdf.plot(), cv2.COLOR_BGR2RGB)
        best_pdf_rgb = cv2.cvtColor(best_pdf.plot(), cv2.COLOR_BGR2RGB)

        fig, axes = plt.subplots(1, 2, figsize=(20, 10))
        axes[0].imshow(early_pdf_rgb)
        axes[0].set_title("Early checkpoint on rasterized PDF")
        axes[0].axis("off")

        axes[1].imshow(best_pdf_rgb)
        axes[1].set_title("Best checkpoint on rasterized PDF")
        axes[1].axis("off")
        plt.tight_layout()
        plt.show()
    else:
        print("Skipping rasterized-PDF comparison (missing PDF or pymupdf).")


### How to read this checkpoint comparison

Typical pattern learners may observe:
- Early checkpoint: fewer correct detections, lower confidence, more wrong classes, more missed symbols.
- Best checkpoint: more stable detections, higher confidence on true objects, fewer obvious mistakes.

Important learning point:
- Improvement is usually larger on in-domain test images than on external rasterized PDF pages.
- This shows two separate effects:
  1. **Training progress** (early -> best checkpoint), and
  2. **Domain gap** (test split -> real external page).

Use this section as a visual proof that model quality evolves over epochs, but deployment performance still depends on data similarity.


## Part 10 — What to do next (project roadmap)

1. **Improve labels iteratively:** Review wrong predictions, then manually correct/add annotations and retrain.
2. **Tiles / sliding windows:** For large sheets, train and infer on **overlapping crops** then merge boxes (NMS across tiles).
3. **Oriented boxes:** For rotated text/symbols, explore **YOLO-OBB** if your tooling supports it.
4. **Combine with OCR:** Detect regions first, then run OCR **inside** each box for readable text (your OCR course notebook).

---
**End of notebook.**
